# D.R.O.N.A. — Bias Detection Benchmark (C2)

Evaluates Contribution C2: rule-based cognitive bias detector.

**Bias archetypes tested:**
- `anchoring` — student fixated on one employer / technology
- `availability` — decision driven by recent anecdotes
- `status_quo` — parental / peer pressure lock-in
- `overconfidence` — inflated skill self-assessment
- `consistency` — prior public commitment prevents reconsideration

**Metrics:** Precision, Recall, F1 per bias type + macro-average.

**No external services needed** — bias detection is pure Python rule-based logic.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from drona.advising.bias_detector import BiasDetector
from drona.utils.logging import setup_logging
setup_logging('WARNING')

bd = BiasDetector()
print('BiasDetector loaded.')
print('Bias types:', [str(bt) for bt in bd._patterns] if hasattr(bd, '_patterns') else 'see BiasDetector source')

## 1. Load Synthetic Test Set

In [ ]:
# Canonical test set — hand-authored, labelled by bias type
# Each item: {text, expected_bias: str | None, label}
TEST_SET = [
    # ── Anchoring ────────────────────────────────────────────────────────────
    {'text': 'I only want to work at Google. Nowhere else will do.', 'expected': 'anchoring', 'label': 'anchor_google'},
    {'text': 'My dream is Facebook, only Facebook, I cannot imagine working anywhere else.', 'expected': 'anchoring', 'label': 'anchor_fb'},
    {'text': 'I am only interested in Python, I refuse to learn anything else.', 'expected': 'anchoring', 'label': 'anchor_py'},
    {'text': 'The only company worth working for is Microsoft.', 'expected': 'anchoring', 'label': 'anchor_ms'},
    {'text': 'I want to explore different companies and roles in Kathmandu.', 'expected': None, 'label': 'clean_explore'},
    # ── Availability ─────────────────────────────────────────────────────────
    {'text': 'My friend just got into machine learning and is earning a lot, so I want to do ML.', 'expected': 'availability', 'label': 'avail_friend_ml'},
    {'text': 'I heard recently that blockchain pays well so I should do blockchain.', 'expected': 'availability', 'label': 'avail_blockchain'},
    {'text': 'Everyone I know is going into data science, so I should too.', 'expected': 'availability', 'label': 'avail_ds'},
    {'text': 'I read recently that AI engineers make a lot of money, so I should do AI.', 'expected': 'availability', 'label': 'avail_ai'},
    {'text': 'I want to research multiple pathways before deciding on a specialisation.', 'expected': None, 'label': 'clean_research'},
    # ── Status quo ───────────────────────────────────────────────────────────
    {'text': 'My parents want me to become a software engineer at a bank and I cannot disappoint them.', 'expected': 'status_quo', 'label': 'sq_parents_bank'},
    {'text': 'My family has decided I should work in Nepal and I cannot change that.', 'expected': 'status_quo', 'label': 'sq_family_nepal'},
    {'text': 'My parents expect me to do a government job and I feel I cannot say no.', 'expected': 'status_quo', 'label': 'sq_govt'},
    {'text': 'My family expects me to follow my elder brother into IT consulting.', 'expected': 'status_quo', 'label': 'sq_brother'},
    {'text': 'I am open to any career that matches my skills and interests.', 'expected': None, 'label': 'clean_open'},
    # ── Overconfidence ───────────────────────────────────────────────────────
    {'text': 'I am an expert in Python and I know everything about machine learning already.', 'expected': 'overconfidence', 'label': 'oc_python_ml'},
    {'text': 'I already know all the skills needed for a senior developer role, I am fully ready.', 'expected': 'overconfidence', 'label': 'oc_senior'},
    {'text': 'I am a master of cybersecurity and know everything I need to know.', 'expected': 'overconfidence', 'label': 'oc_cybersec'},
    {'text': 'I am an expert in everything and I think I am ready for any job.', 'expected': 'overconfidence', 'label': 'oc_all'},
    {'text': 'I would like to improve my skills in data analysis and cloud computing.', 'expected': None, 'label': 'clean_improve'},
    # ── Consistency ──────────────────────────────────────────────────────────
    {'text': 'I have told everyone I will be a data scientist. I cannot change now.', 'expected': 'consistency', 'label': 'cons_ds_public'},
    {'text': 'I have already committed publicly to becoming an Android developer so I cannot switch.', 'expected': 'consistency', 'label': 'cons_android'},
    {'text': 'I told my college friends I would work abroad. Changing would be embarrassing.', 'expected': 'consistency', 'label': 'cons_abroad'},
    {'text': 'I already said I will join a startup. Going back on that would disappoint people.', 'expected': 'consistency', 'label': 'cons_startup'},
    {'text': 'What career paths are available for BSc Computing graduates in Nepal?', 'expected': None, 'label': 'clean_generic'},
]

print(f'Test set: {len(TEST_SET)} items')
biased = sum(1 for x in TEST_SET if x['expected'] is not None)
clean  = sum(1 for x in TEST_SET if x['expected'] is None)
print(f'  Biased: {biased}  |  Clean: {clean}')

## 2. Run Detector

In [ ]:
rows = []
for item in TEST_SET:
    flags = bd.detect(item['text'])
    detected_types = [str(f.bias_type) for f in flags]
    expected = item['expected']
    
    tp = expected is not None and expected in detected_types
    fp = any(dt != expected for dt in detected_types)
    fn = expected is not None and expected not in detected_types
    tn = expected is None and not detected_types
    
    rows.append({
        'label': item['label'],
        'expected': expected or 'none',
        'detected': ', '.join(detected_types) if detected_types else 'none',
        'correct': (expected in detected_types) if expected else (not detected_types),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'n_flags': len(flags),
    })

df = pd.DataFrame(rows)
print(f'Overall accuracy: {df["correct"].mean():.1%} ({df["correct"].sum()}/{len(df)})')
print()
print(df[['label', 'expected', 'detected', 'correct']].to_string(index=False))

## 3. Per-Bias Precision / Recall / F1

In [ ]:
BIAS_TYPES = ['anchoring', 'availability', 'status_quo', 'overconfidence', 'consistency']

per_bias = {}
for bias_type in BIAS_TYPES:
    # Positive: expected == bias_type
    # Predicted positive: bias_type in detected
    relevant = df[df['expected'] == bias_type]
    all_pred_positive = df[df['detected'].str.contains(bias_type, na=False)]
    
    tp = len(relevant[relevant['detected'].str.contains(bias_type, na=False)])
    fp = len(all_pred_positive[all_pred_positive['expected'] != bias_type])
    fn = len(relevant[~relevant['detected'].str.contains(bias_type, na=False)])
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    per_bias[bias_type] = {'precision': precision, 'recall': recall, 'f1': f1,
                           'tp': tp, 'fp': fp, 'fn': fn, 'n_test': len(relevant)}

metrics_df = pd.DataFrame(per_bias).T
macro_f1  = metrics_df['f1'].mean()
macro_prec = metrics_df['precision'].mean()
macro_rec  = metrics_df['recall'].mean()

print(metrics_df[['precision', 'recall', 'f1', 'n_test']].round(3).to_string())
print(f'\nMacro Precision: {macro_prec:.3f}')
print(f'Macro Recall:    {macro_rec:.3f}')
print(f'Macro F1:        {macro_f1:.3f}  (target ≥ 0.85, acceptable ≥ 0.80)')
print(f'C2 status:       {"✅ PASS" if macro_f1 >= 0.80 else "❌ FAIL"}')

## 4. Visualise Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-bias F1
x = np.arange(len(BIAS_TYPES))
width = 0.25
axes[0].bar(x - width, metrics_df['precision'], width, label='Precision', color='steelblue')
axes[0].bar(x,         metrics_df['recall'],    width, label='Recall',    color='#FF9800')
axes[0].bar(x + width, metrics_df['f1'],        width, label='F1',        color='#4CAF50')
axes[0].axhline(0.85, color='red', linestyle='--', linewidth=1, label='Target F1=0.85')
axes[0].axhline(0.80, color='orange', linestyle=':', linewidth=1, label='Acceptable F1=0.80')
axes[0].set_xticks(x)
axes[0].set_xticklabels(BIAS_TYPES, rotation=20, ha='right')
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel('Score')
axes[0].set_title('C2 — Per-Bias Precision / Recall / F1')
axes[0].legend(fontsize=8)

# Confusion summary per bias
tp_vals = [per_bias[b]['tp'] for b in BIAS_TYPES]
fn_vals = [per_bias[b]['fn'] for b in BIAS_TYPES]
fp_vals = [per_bias[b]['fp'] for b in BIAS_TYPES]
x2 = np.arange(len(BIAS_TYPES))
axes[1].bar(x2, tp_vals, label='TP', color='#4CAF50')
axes[1].bar(x2, fn_vals, bottom=tp_vals, label='FN', color='#F44336')
axes[1].bar(x2, fp_vals, bottom=[t+f for t,f in zip(tp_vals, fn_vals)], label='FP', color='#FF9800')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(BIAS_TYPES, rotation=20, ha='right')
axes[1].set_ylabel('Count')
axes[1].set_title('Confusion Breakdown per Bias Type')
axes[1].legend()

plt.suptitle(f'Bias Detection Benchmark — Macro F1={macro_f1:.3f}', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/evaluation/c2_bias_detection.png', dpi=150, bbox_inches='tight')
plt.show()

# Save JSON
os.makedirs('../data/evaluation', exist_ok=True)
result = {
    'macro_f1': float(macro_f1),
    'macro_precision': float(macro_prec),
    'macro_recall': float(macro_rec),
    'per_bias': {b: {k: float(v) if isinstance(v, float) else int(v)
                     for k, v in per_bias[b].items()} for b in BIAS_TYPES},
    'n_queries': len(TEST_SET),
    'c2_pass': macro_f1 >= 0.80,
}
with open('../data/evaluation/c2_bias_detection.json', 'w') as f:
    json.dump(result, f, indent=2)
print('Results saved to data/evaluation/c2_bias_detection.json')

## 5. Error Analysis

In [ ]:
errors = df[~df['correct']]
if len(errors) == 0:
    print('All test cases classified correctly!')
else:
    print(f'{len(errors)} misclassified cases:')
    for _, row in errors.iterrows():
        print(f"  [{row['label']:25s}] expected={row['expected']:15s} got={row['detected']}")
    print()
    print('Review bias_detector.py keyword patterns for the misclassified bias types.')